# z608 - Features de stock (Etapa 9)
Agrega `stock_final` sobre FE604 (mejor baseline confirmado, 0.258). Cobertura parcial: `tb_stocks.txt` tiene 13691 filas vs ~31522 de la tabla principal -- va a haber muchos nulls, LightGBM los maneja nativamente sin imputar.

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'FE609',
    'features_path': '/home/ds/datasets/tb_features_FE604.parquet',
    'stocks_path': '/home/ds/datasets/tb_stocks.txt'
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/FE609


## 1. Cargar stock y unir con FE604

In [4]:
tb_stocks = pl.read_csv(PARAM['stocks_path'], separator="\t")

df = pl.read_parquet(PARAM['features_path'])
df = df.join(tb_stocks, on=["product_id", "periodo"], how="left")

print("nulls en stock_final:", df["stock_final"].null_count(), "de", df.height)

nulls en stock_final: 17831 de 31522


## 2. Ratio stock / venta
`stock_final(p) / tn(p)` -- cuantos periodos de cobertura tiene el stock actual respecto a la venta del mes. Alto: sobrestockeado. Bajo: cerca de agotarse.

In [5]:
df = df.with_columns(
    (pl.col("stock_final") / (pl.col("tn") + 1e-6)).alias("ratio_stock_tn")
)

## 3. Guardar

In [6]:
salida = os.path.join(ruta, "tb_features_FE609.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/FE609/tb_features_FE609.parquet
(31522, 74)
